In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from google.colab import drive

drive.mount('/content/drive')

feature_df = pd.read_pickle("/content/drive/MyDrive/SmartRail/features_full.pkl")

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = feature_df.drop(columns=["failure"])
y = feature_df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Setup done")

Mounted at /content/drive
Setup done


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

sample_idx_gb = np.random.choice(len(X_train), 200000, replace=False)
X_train_gb = X_train.iloc[sample_idx_gb]
y_train_gb = y_train.iloc[sample_idx_gb]

gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb.fit(X_train_gb, y_train_gb)

auc_gb = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])
print(f"Gradient Boosting AUC: {auc_gb:.4f}")

joblib.dump(gb, "/content/drive/MyDrive/SmartRail/gradient_boosting.pkl")

Gradient Boosting AUC: 1.0000


['/content/drive/MyDrive/SmartRail/gradient_boosting.pkl']

In [ ]:
with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["Gradient Boosting"] = {"auc": 1.0000}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

In [ ]:
!pip install xgboost -q

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                     scale_pos_weight=scale_pos_weight, random_state=42,
                     eval_metric='logloss')
xgb.fit(X_train, y_train)

auc_xgb = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
print(f"XGBoost AUC: {auc_xgb:.4f}")

joblib.dump(xgb, "/content/drive/MyDrive/SmartRail/xgboost.pkl")

XGBoost AUC: 1.0000


['/content/drive/MyDrive/SmartRail/xgboost.pkl']

In [ ]:
with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["XGBoost"] = {"auc": 1.0000}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

In [ ]:
xgb_importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
xgb_importances.sort_values(ascending=False).head(10)

,0
kmeans_cluster,0.962139
TP2_roll_min,0.019982
pca_distance_score,0.005462
DV_pressure_lag1,0.001852
Motor_current_roll_max,0.001479
DV_pressure_roll_min,0.001446
DV_pressure_roll_max,0.001057
Oil_temperature_roll_min,0.000960
DV_pressure_roll_mean,0.000686
H1_roll_mean,0.000558


In [ ]:
X_train_noclus = X_train.drop(columns=["kmeans_cluster"])
X_test_noclus = X_test.drop(columns=["kmeans_cluster"])

xgb_v2 = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                        scale_pos_weight=scale_pos_weight, random_state=42,
                        eval_metric='logloss')
xgb_v2.fit(X_train_noclus, y_train)

auc_v2 = roc_auc_score(y_test, xgb_v2.predict_proba(X_test_noclus)[:, 1])
print(f"XGBoost WITHOUT kmeans_cluster: {auc_v2:.4f}")

XGBoost WITHOUT kmeans_cluster: 1.0000


In [ ]:
feature_df_sorted = feature_df.sort_index()

split_point = int(len(feature_df_sorted) * 0.8)
train_df = feature_df_sorted.iloc[:split_point]
test_df = feature_df_sorted.iloc[split_point:]

X_train_time = train_df.drop(columns=["failure"])
y_train_time = train_df["failure"]
X_test_time = test_df.drop(columns=["failure"])
y_test_time = test_df["failure"]

print(y_train_time.value_counts())
print(y_test_time.value_counts())

failure
0    1137692
1      29863
Name: count, dtype: int64
failure
0    291889
Name: count, dtype: int64


In [ ]:
feature_df_sorted = feature_df.sort_index()

block_size = 10000
feature_df_sorted["block"] = np.arange(len(feature_df_sorted)) // block_size

test_blocks = feature_df_sorted["block"] % 5 == 0

train_df = feature_df_sorted[~test_blocks].drop(columns=["block"])
test_df = feature_df_sorted[test_blocks].drop(columns=["block"])

X_train_time = train_df.drop(columns=["failure"])
y_train_time = train_df["failure"]
X_test_time = test_df.drop(columns=["failure"])
y_test_time = test_df["failure"]

print(y_train_time.value_counts())
print(y_test_time.value_counts())

failure
0    1134132
1      25868
Name: count, dtype: int64
failure
0    295449
1      3995
Name: count, dtype: int64


In [ ]:
xgb_v3 = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                        scale_pos_weight=(y_train_time==0).sum()/(y_train_time==1).sum(),
                        random_state=42, eval_metric='logloss')
xgb_v3.fit(X_train_time, y_train_time)

auc_v3 = roc_auc_score(y_test_time, xgb_v3.predict_proba(X_test_time)[:, 1])
print(f"XGBoost (block-wise time split): {auc_v3:.4f}")

XGBoost (block-wise time split): 0.9997


In [ ]:
joblib.dump(xgb_v3, "/content/drive/MyDrive/SmartRail/xgboost_final.pkl")

import pickle
with open("/content/drive/MyDrive/SmartRail/time_split_data.pkl", "wb") as f:
    pickle.dump({
        "X_train_time": X_train_time, "y_train_time": y_train_time,
        "X_test_time": X_test_time, "y_test_time": y_test_time
    }, f)

with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["XGBoost_final_timesplit"] = {"auc": 0.9997}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

print("Saved")

Saved


In [ ]:
print("Progress summary saved in results.json — checking now:")
with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    print(json.load(f))

Progress summary saved in results.json — checking now:
{'Logistic Regression': {'precision': 0.57, 'recall': 1.0, 'f1': 0.73}, 'Naive Bayes': {'precision': 0.47, 'recall': 0.99, 'f1': 0.63, 'auc': 0.9859}, 'KNN': {'precision': 0.96, 'recall': 0.96, 'f1': 0.96, 'auc': 0.9971}, 'SVM': {'precision': 0.76, 'recall': 0.99, 'f1': 0.86, 'auc': 0.9993}, 'Decision Tree': {'precision': 0.86, 'recall': 1.0, 'f1': 0.92, 'auc': 0.9991}, 'Bagging': {'auc': 0.9999}, 'Random Forest': {'auc': 1.0}, 'AdaBoost': {'auc': 0.9999}, 'Gradient Boosting': {'auc': 1.0}, 'XGBoost': {'auc': 1.0}, 'XGBoost_final_timesplit': {'auc': 0.9997}}
